# Title/Abstract Screening with AgenticWorkflow

This tutorial demonstrates how to use `TitleAbstractReviewer` with `AgenticWorkflow` to screen a dataset of 978 medical AI articles. We will:

1. Load a real dataset of radiology/AI research articles
2. Screen articles with a `TitleAbstractReviewer` using agentic capabilities (memory, flagging)
3. Run a two-round workflow with an expert second-pass reviewer on a different model
4. Inspect the memory and flags generated during the review

**Requirements:** `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` environment variables must be set.

## Setup

In [1]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from pathlib import Path
from lattereview.agentic import TitleAbstractReviewer, AgenticWorkflow

## Load the Dataset

The dataset contains 978 articles from the medical AI literature. Each row has a `title`, `abstract`, `DOI`, and several annotation columns (`study_type`, `clinical_application`, `organ`, `modality`, `cardiovascular`, etc.).

Our goal: screen for articles about **AI/deep learning applied to cardiovascular imaging**.

In [2]:
df = pd.read_csv("data.csv")
print(f"Dataset: {len(df)} articles")
print(f"Columns: {list(df.columns)}")
df[["title", "abstract"]].head(3)

Dataset: 978 articles
Columns: ['title', 'abstract', 'DOI', 'study_type', 'clinical_application', 'organ', 'modality', 'cardiovascular', 'task', 'deep_learning', 'external_validation']


,title,abstract
0,(18)F-FDG PET/CT Uptake Classification in Lymp...,Background Fluorine 18 ((18)F)-fluorodeoxygluc...
1,(18)F-FDG-PET/CT Whole-Body Imaging Lung Tumor...,Under the background of (18)F-FDG-PET/CT multi...
2,3-D Convolutional Neural Networks for Automati...,Deep two-dimensional (2-D) convolutional neura...


## Round 1: Initial Screening

We create a `TitleAbstractReviewer` with clear inclusion and exclusion criteria. The reviewer uses:
- `managing-memory` — to remember patterns across articles (e.g., "studies mentioning 'echocardiography' are likely cardiovascular")
- `flagging-items` — to flag borderline articles for human review
- `max_iterations=10` — allowing the agent multiple reasoning steps
- `agentic_effort="medium"` — balanced depth of analysis

For this tutorial, we process only the first 5 rows to keep costs and time manageable.

In [3]:
screener = TitleAbstractReviewer(
    name="CardioScreener",
    backstory=(
        "You are a systematic review expert specializing in cardiovascular imaging "
        "and artificial intelligence. You have extensive experience screening "
        "radiology and cardiology AI literature for systematic reviews."
    ),
    model="openai:gpt-5.4-mini",
    inclusion_criteria=(
        "Studies that apply AI, machine learning, or deep learning methods "
        "to cardiovascular imaging tasks (echocardiography, cardiac MRI, "
        "cardiac CT, coronary angiography, nuclear cardiology). "
        "The study must involve image analysis or image-based diagnosis."
    ),
    exclusion_criteria=(
        "Non-imaging studies (e.g., ECG-only, EHR-only, genomics). "
        "Animal studies or phantom-only studies. "
        "Studies focused on non-cardiovascular organs even if using AI imaging. "
        "Review articles, editorials, or commentaries without original data."
    ),
    max_iterations=10,
    agentic_effort="medium",
    skills=["managing-memory", "flagging-items"],
)

print(f"Reviewer: {screener.name}")
print(f"Skills: {screener.skills}")
print(f"Max iterations: {screener.max_iterations}")

Reviewer: CardioScreener
Skills: ['managing-memory', 'flagging-items']
Max iterations: 10


In [4]:
import shutil

WORKING_DIR = Path("./screening_output")
if WORKING_DIR.exists():
    shutil.rmtree(WORKING_DIR)

workflow_round1 = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [screener],
            "text_inputs": ["title", "abstract"],
        }
    ],
    working_dir=WORKING_DIR,
    verbose=True,
)

# Process only the first 5 rows for this tutorial
subset_df = df.head(5).copy()
result_df = await workflow_round1(subset_df)

print(f"\nProcessed {len(result_df)} articles")
print(f"Total cost: ${workflow_round1.total_cost:.4f}")


====== Starting review round A (1/1) ======

Processing 5 eligible rows
Running reviewer: CardioScreener (5 items)


Reviewer CardioScreener: 5 successful, 0 failed
Columns after CardioScreener: ['title', 'abstract', 'DOI', 'study_type', 'clinical_application', 'organ', 'modality', 'cardiovascular', 'task', 'deep_learning', 'external_validation', 'round-A_CardioScreener_output', 'round-A_CardioScreener_reasoning', 'round-A_CardioScreener_decision', 'round-A_CardioScreener_certainty']
Saved snapshot after round A

Workflow complete. Total cost: $0.0000

Processed 5 articles
Total cost: $0.0000


## Inspect Round 1 Results

The workflow adds columns following the naming convention `round-{ID}_{reviewer_name}_{field}`. For `TitleAbstractReviewer`, the output fields are:

| Column | Description |
|--------|-------------|
| `round-A_CardioScreener_reasoning` | The reviewer's reasoning for the decision |
| `round-A_CardioScreener_decision` | Include/exclude score (1-5 Likert scale) |
| `round-A_CardioScreener_certainty` | Confidence level (0-100) |
| `round-A_CardioScreener_output` | Full output dict |

In [5]:
output_cols = [c for c in result_df.columns if c.startswith("round-A")]
print("Output columns:", output_cols)
print()

# Display decisions with titles
for idx, row in result_df.iterrows():
    title = row["title"][:80]
    decision = row.get("round-A_CardioScreener_decision", "N/A")
    certainty = row.get("round-A_CardioScreener_certainty", "N/A")
    print(f"[{decision}] (certainty: {certainty}) {title}...")

Output columns: ['round-A_CardioScreener_output', 'round-A_CardioScreener_reasoning', 'round-A_CardioScreener_decision', 'round-A_CardioScreener_certainty']

[exclude] (certainty: 99) (18)F-FDG PET/CT Uptake Classification in Lymphoma and Lung Cancer by Using Deep...
[1] (certainty: 99) (18)F-FDG-PET/CT Whole-Body Imaging Lung Tumor Diagnostic Model: An Ensemble E-R...
[exclude] (certainty: 99) 3-D Convolutional Neural Networks for Automatic Detection of Pulmonary Nodules i...
[exclude] (certainty: 99) 3D CNN with Visual Insights for Early Detection of Lung Cancer Using Gradient-We...
[exclude] (certainty: 99) 3D deep learning based classification of pulmonary ground glass opacity nodules ...


## Round 2: Expert Review with a Different Model

Now we add a second round where an expert reviewer (using `anthropic:claude-sonnet-4-6`) re-evaluates articles that were included (decision >= 3) in Round A. This provides a second opinion from a different model.

We set up a complete two-round workflow from scratch.

In [6]:
# First-pass screener (same as before)
first_screener = TitleAbstractReviewer(
    name="Screener",
    backstory=(
        "You are a research assistant screening articles on cardiovascular AI imaging. "
        "Be inclusive in borderline cases — it is better to include a questionable "
        "article than to miss a relevant one."
    ),
    model="openai:gpt-5.4-mini",
    inclusion_criteria=(
        "AI/deep learning applied to cardiovascular imaging "
        "(echocardiography, cardiac MRI, cardiac CT, coronary angiography)"
    ),
    exclusion_criteria=(
        "Non-imaging studies, animal studies, reviews without original data"
    ),
    max_iterations=10,
    agentic_effort="medium",
    skills=["managing-memory"],
)

# Second-pass expert using a different model
expert_screener = TitleAbstractReviewer(
    name="Expert",
    backstory=(
        "You are a senior cardiologist and AI researcher with 20 years of experience. "
        "You carefully evaluate whether studies meet rigorous inclusion criteria. "
        "You are more conservative than initial screeners."
    ),
    model="anthropic:claude-sonnet-4-6",
    inclusion_criteria=(
        "Studies with rigorous AI/ML methodology applied to cardiovascular "
        "diagnostic imaging. Must present original research with quantitative results."
    ),
    exclusion_criteria=(
        "Weak methodology, non-diagnostic applications, purely technical AI papers "
        "without clinical imaging data, non-cardiovascular imaging"
    ),
    max_iterations=10,
    agentic_effort="medium",
    skills=["managing-memory", "flagging-items"],
)

In [7]:
import shutil

WORKING_DIR_2 = Path("./screening_two_round")
if WORKING_DIR_2.exists():
    shutil.rmtree(WORKING_DIR_2)

def include_filter(row):
    """Include articles that the first screener rated 3 or higher (borderline to definite include)."""
    decision = row.get("round-A_Screener_decision")
    if decision is None or decision == "" or pd.isna(decision):
        return False
    try:
        return int(decision) >= 3
    except (ValueError, TypeError):
        # If decision is text like "include"/"exclude", include anything that's not explicitly "exclude"
        return str(decision).strip().lower() not in ("exclude", "1", "2")

workflow_2round = AgenticWorkflow(
    workflow_schema=[
        {
            "round": "A",
            "reviewers": [first_screener],
            "text_inputs": ["title", "abstract"],
        },
        {
            "round": "B",
            "reviewers": [expert_screener],
            "text_inputs": ["title", "abstract", "round-A_Screener_output"],
            "filter": include_filter,
        },
    ],
    working_dir=WORKING_DIR_2,
    verbose=True,
)

subset_df_2 = df.head(5).copy()
result_df_2 = await workflow_2round(subset_df_2)

print(f"\nTotal cost: ${workflow_2round.total_cost:.4f}")


====== Starting review round A (1/2) ======

Processing 5 eligible rows
Running reviewer: Screener (5 items)


Reviewer Screener: 5 successful, 0 failed
Columns after Screener: ['title', 'abstract', 'DOI', 'study_type', 'clinical_application', 'organ', 'modality', 'cardiovascular', 'task', 'deep_learning', 'external_validation', 'round-A_Screener_output', 'round-A_Screener_reasoning', 'round-A_Screener_decision', 'round-A_Screener_certainty']
Saved snapshot after round A

====== Starting review round B (2/2) ======

Skipping round B — no eligible rows

Workflow complete. Total cost: $0.0000

Total cost: $0.0000


In [8]:
# Compare Round A and Round B results
all_output_cols = [c for c in result_df_2.columns if c.startswith("round-")]
print("All output columns:", all_output_cols)
print()

for idx, row in result_df_2.iterrows():
    title = row["title"][:60]
    round_a = row.get("round-A_Screener_decision", "N/A")
    round_b = row.get("round-B_Expert_decision", "--")
    print(f"Round A: {round_a} | Round B: {round_b} | {title}...")

All output columns: ['round-A_Screener_output', 'round-A_Screener_reasoning', 'round-A_Screener_decision', 'round-A_Screener_certainty']

Round A: exclude | Round B: -- | (18)F-FDG PET/CT Uptake Classification in Lymphoma and Lung ...
Round A: exclude | Round B: -- | (18)F-FDG-PET/CT Whole-Body Imaging Lung Tumor Diagnostic Mo...
Round A: exclude | Round B: -- | 3-D Convolutional Neural Networks for Automatic Detection of...
Round A: exclude | Round B: -- | 3D CNN with Visual Insights for Early Detection of Lung Canc...
Round A: 1 | Round B: -- | 3D deep learning based classification of pulmonary ground gl...


## Inspect Memory and Flags

When `managing-memory` and `flagging-items` skills are enabled, the agents save files to the working directory. Let's inspect what the reviewers learned and flagged.

In [9]:
import json
import os

# Show the directory structure
for wd, label in [(WORKING_DIR, "Round 1 Output"), (WORKING_DIR_2, "Two-Round Output")]:
    if wd.exists():
        print(f"\n=== {label} ({wd}) ===")
        for root, dirs, files in os.walk(wd):
            level = root.replace(str(wd), "").count(os.sep)
            indent = "  " * level
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = "  " * (level + 1)
            for f in files:
                print(f"{sub_indent}{f}")


=== Round 1 Output (screening_output) ===
screening_output/
  run_metadata.json
  output/
    final.parquet
    after_round_A.parquet
  round_A/
    agent_CardioScreener/
      logs/
        item_A-2.jsonl
        item_A-0.jsonl
        item_A-3.jsonl
        item_A-4.jsonl
        item_A-1.jsonl
      memory/
      flags/
      results/
        item_A-4.json
        item_A-3.json
        item_A-2.json
        item_A-1.json
        item_A-0.json

=== Two-Round Output (screening_two_round) ===
screening_two_round/
  run_metadata.json
  output/
    final.parquet
    after_round_A.parquet
  round_A/
    agent_Screener/
      logs/
        item_A-2.jsonl
        item_A-0.jsonl
        item_A-3.jsonl
        item_A-4.jsonl
        item_A-1.jsonl
      memory/
      flags/
      results/
        item_A-4.json
        item_A-3.json
        item_A-2.json
        item_A-1.json
        item_A-0.json


In [10]:
# Read the memory index if it exists
memory_paths = list(WORKING_DIR_2.glob("**/memory/_index.json"))
for mp in memory_paths:
    print(f"\n--- Memory index: {mp.relative_to(WORKING_DIR_2)} ---")
    with open(mp) as f:
        index = json.load(f)
    memories = index.get("memories", index) if isinstance(index, dict) else index
    if isinstance(memories, list):
        for entry in memories:
            print(f"  [{entry.get('id', '?')}] {entry.get('brief', entry.get('title', ''))}")
    else:
        print(f"  {index}")

# Read the flags if they exist
flag_paths = list(WORKING_DIR_2.glob("**/flags/flags.json"))
for fp in flag_paths:
    print(f"\n--- Flags: {fp.relative_to(WORKING_DIR_2)} ---")
    with open(fp) as f:
        flags = json.load(f)
    if flags:
        for flag in flags:
            status = "resolved" if flag.get("resolved") else "unresolved"
            print(f"  [{status}] {flag.get('item_id', '?')}: {flag.get('reason', '')}")
    else:
        print("  No items flagged.")

## Clean Up

In [11]:
import shutil

for wd in [WORKING_DIR, WORKING_DIR_2]:
    if wd.exists():
        shutil.rmtree(wd)
        print(f"Removed {wd}")

Removed screening_output
Removed screening_two_round
